In [1]:
import os
import torch
import numpy as np

from PIL import Image
from sklearn.model_selection import train_test_split

from torchvision.datasets import ImageFolder
from torchvision import transforms

from torch.utils.data import DataLoader, random_split

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification
)

from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from tqdm import tqdm

d:\DreamProject\backend\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [3]:
DATASET_PATH = "dataset"

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),

    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

Load datset


In [5]:
dataset = ImageFolder(
    DATASET_PATH,
    transform=train_transform
)

print(dataset.classes)

['Acrylic', 'ColorPencil', 'GraphitePencil', 'Oil', 'Pastel', 'Photo', 'Watercolor']


SHAPE OF EACH FOLDER

In [6]:
import os

dataset_path = "dataset"

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

print("=" * 40)
print("Dataset Summary")
print("=" * 40)

total_images = 0

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        count = sum(
            1
            for file in os.listdir(folder_path)
            if file.lower().endswith(image_extensions)
        )

        total_images += count

        print(f"{folder:<20} : {count}")

print("=" * 40)
print(f"Total Images       : {total_images}")
print("=" * 40)

Dataset Summary
Acrylic              : 196
ColorPencil          : 137
GraphitePencil       : 142
Oil                  : 200
Pastel               : 200
Photo                : 125
Watercolor           : 170
Total Images       : 1170


Split train and test

In [7]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

val_dataset.dataset.transform = val_transform

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [9]:
MODEL_NAME = "google/siglip-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(dataset.classes),
    ignore_mismatched_sizes=True
)

model.to(device)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 208/208 [00:00<00:00, 1200.87it/s]
[transformers] SiglipForImageClassification LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_at

SiglipForImageClassification(
  (vision_model): SiglipVisionModel(
    (embeddings): SiglipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
      (position_embedding): Embedding(196, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768,

In [10]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

criterion = CrossEntropyLoss()

In [11]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    correct = 0

    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=images)

        logits = outputs.logits

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

    acc = correct / total

    print(f"Epoch {epoch+1}")

    print("Loss:", total_loss)

    print("Train Accuracy:", acc)

  0%|          | 0/59 [00:00<?, ?it/s]

100%|██████████| 59/59 [24:45<00:00, 25.17s/it]


Epoch 1
Loss: 73.42968851327896
Train Accuracy: 0.5544871794871795


100%|██████████| 59/59 [23:37<00:00, 24.02s/it]


Epoch 2
Loss: 19.198490157723427
Train Accuracy: 0.8942307692307693


...

Epoch 3
Loss: 5.218828071840107
Train Accuracy: 0.9754273504273504


100%|██████████| 59/59 [21:29<00:00, 21.86s/it]


Epoch 4
Loss: 1.4817398536251858
Train Accuracy: 0.9957264957264957


100%|██████████| 59/59 [21:40<00:00, 22.05s/it]

Epoch 5
Loss: 0.9816544007044286
Train Accuracy: 0.9978632478632479


In [12]:
model.eval()

correct = 0

total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(pixel_values=images)

        preds = outputs.logits.argmax(dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

print("Validation Accuracy:", correct/total)

Validation Accuracy: 0.7948717948717948


In [13]:
torch.save(
    model.state_dict(),
    "medium_classifier.pth"
)

print("Saved!")

Saved!


In [14]:
import json

with open("labels.json","w") as f:
    json.dump(dataset.classes,f)

print(dataset.classes)

['Acrylic', 'ColorPencil', 'GraphitePencil', 'Oil', 'Pastel', 'Photo', 'Watercolor']
